# CS 297 - Experiment 1
### Description: Can a small, open weight model reliably detect noise in an image and determine whether denoising is required?
### Ethan Machado


## Experiment design details:

Classification problems:

1.   Noise evaluation task - binary class problem (0 = clean; 1 = augmented)
2.   Noise severity task - multi-class problem (0 = clean; 1 = weak; 2 = medium; 3 = strong)
2.   Routing decision task - (clean, weak --> no denoising; medium, strong --> yes denoising)

Pipeline:

1.   Feed in images from dataset as-is with no preprocessing.
2.   Evaluate the model's performance using zero-shot prompting.
3.   Evaluate the model's performance using few-shot prompting.

Evaluation criteria:

For the noise evaluation task: Accuracy, Precision, Recall, Macro-F1, Confusion matrix.

For the noise severity task: MAE loss function, accuracy, Precision, Recall, Macro-F1, Confusion matrix.

For the routing decision task: Routing accuracy, Precision, Recall, Macro-F1, Confusion matrix.

**Note: There should be an additional test performed to check the stability of decision-making by processing images multiple times during each experiment.**










In [1]:
# Google Drive mounting

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/04_F2026/CS297/experiment1'
os.chdir(PROJECT_PATH)

# allowing for python script imports

import sys
sys.path.append(os.path.join(PROJECT_PATH, 'src'))

print(f"\nCurrent working directory:", os.getcwd())

HF_CACHE_PATH = '/content/drive/MyDrive/04_F2026/CS297/experiment1/hf_cache'
os.environ['HF_HOME'] = HF_CACHE_PATH

print(f"\nHF_HOME environment variable set to:", os.environ['HF_HOME'])


Mounted at /content/drive

Current working directory: /content/drive/MyDrive/04_F2026/CS297/experiment1

HF_HOME environment variable set to: /content/drive/MyDrive/04_F2026/CS297/experiment1/hf_cache


In [5]:
# imports

import torch
import pandas as pd
import numpy as np
from dotenv import load_dotenv


!pip install -q transformers accelerate datasets bitsandbytes

In [6]:
# dataset labeling for all 3 experiments

# define unified mapping configuration
config = {
    "/content/drive/MyDrive/04_F2026/CS297/experiment1/data/files/images": {
        "augmentation": "none",
        "severity_label": 0,
        "noise_label": 0,
        "route_label": 0
    },
    "/content/drive/MyDrive/04_F2026/CS297/experiment1/data/files/augmentation/1": {
        "augmentation": "weak",
        "severity_label": 1,
        "noise_label": 1,
        "route_label": 0
    },
    "/content/drive/MyDrive/04_F2026/CS297/experiment1/data/files/augmentation/2": {
        "augmentation": "medium",
        "severity_label": 2,
        "noise_label": 1,
        "route_label": 1
    },
    "/content/drive/MyDrive/04_F2026/CS297/experiment1/data/files/augmentation/3": {
        "augmentation": "strong",
        "severity_label": 3,
        "noise_label": 1,
        "route_label": 1
    }
}

image_data = []
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp')

# iterate through each directory and compile image data
for folder, mappings in config.items():
    if os.path.exists(folder):
        for file in os.listdir(folder):
            if file.lower().endswith(valid_extensions):
                file_path = os.path.join(folder, file)
                image_data.append({
                    "file_path": file_path,
                    "file_name": file,
                    "augmentation": mappings["augmentation"],
                    "severity_label": mappings["severity_label"],
                    "noise_label": mappings["noise_label"],
                    "route_label": mappings["route_label"]
                })
    else:
        print(f"Warning: Path not found: {folder}")

# create a single unified pandas DataFrame
df_experiment = pd.DataFrame(image_data)

# display summary statistics and the head of the DataFrame
print(f"Total images labeled: {len(df_experiment)}")
if not df_experiment.empty:
    print("\nValue counts by augmentation category:")
    print(df_experiment['augmentation'].value_counts())

df_experiment.head()

Total images labeled: 200

Value counts by augmentation category:
augmentation
none      50
weak      50
medium    50
strong    50
Name: count, dtype: int64


,file_path,file_name,augmentation,severity_label,noise_label,route_label
0,/content/drive/MyDrive/04_F2026/CS297/experime...,10.jpg,none,0,0,0
1,/content/drive/MyDrive/04_F2026/CS297/experime...,1.jpg,none,0,0,0
2,/content/drive/MyDrive/04_F2026/CS297/experime...,11.jpg,none,0,0,0
3,/content/drive/MyDrive/04_F2026/CS297/experime...,0.jpg,none,0,0,0
4,/content/drive/MyDrive/04_F2026/CS297/experime...,19.jpg,none,0,0,0


In [ ]:
# configuration of HF login

from openai import OpenAI

# secrets using Colab UI

# from google.colab import userdata
# api_key = userdata.get('HF_TOKEN')

# secrets using VS Code

api_key = os.environ.get('HF_TOKEN')

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=api_key,
)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
# HF imports

from transformers import AutoProcessor, AutoModelForMultimodalLM

In [ ]:
# initializing Qwen3.5 0.8b

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-0.8B")
model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-0.8B", device_map="auto")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))